In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

# Carga de los ficheros con los indicadores demográricos básicos

Se han estructurado los datos en la carpeta de inputs para la dimensión demográfica de forma que se dispone de un fichero CSV a nivel
de provincia. Así pues, se han definido una función en las utilidades que se encargan de cargar y dar una limpieza inicial a los datos.

Los datos se obtienen del atlas de distribución de renta de los hogares:
https://www.ine.es/dynt3/inebase/index.htm?padre=12385&capsel=12384

In [2]:
path = os.path.join(DATA_INPUTS_DD, "Indicadores demograficos")

indicadores = carga_datos_ine(path)

# Veo una muestra de su estructura y contenido
print(indicadores.info())
indicadores.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 224973 entries, 126 to 515591
Data columns (total 7 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   Municipios                224973 non-null  object
 1   Distritos                 224973 non-null  object
 2   Secciones                 224973 non-null  object
 3   Indicadores demográficos  224973 non-null  object
 4   Periodo                   224973 non-null  int64 
 5   Total                     198755 non-null  object
 6   Provincia                 224973 non-null  object
dtypes: int64(1), object(6)
memory usage: 13.7+ MB
None


,Municipios,Distritos,Secciones,Indicadores demográficos,Periodo,Total,Provincia
69189,09059 Burgos,0905908 Burgos distrito 08,0905908002 Burgos sección 08002,Porcentaje de población menor de 18 años,2017,"17,6",Burgos
439742,47186 Valladolid,4718604 Valladolid distrito 04,4718604007 Valladolid sección 04007,Edad media de la población,2021,"46,4",Valladolid
47160,05244 Tormellas,0524401 Tormellas distrito 01,0524401001 Tormellas sección 01001,Porcentaje de hogares unipersonales,2023,"59,1",Avila
335007,40107 Labajos,4010701 Labajos distrito 01,4010701001 Labajos sección 01001,Porcentaje de hogares unipersonales,2023,"38,8",Segovia
188702,24210 Villagatón,2421001 Villagatón distrito 01,2421001001 Villagatón sección 01001,Porcentaje de población menor de 18 años,2015,"4,6",Leon


# Estandarización del dataframe de datos del INE

Como se puede observar, el fichero csv de datos del INE tiene un formato poco amigable para el tratamiento de los datos. En lugar de tener una fila
por cada par sección-año y varias columnas (una por factor), tiene múltiples filas con distintos indicadores para una misma sección, lo que resulta
complejo de tratar. Además, se observa como se mezcla el código del municipio, distrito y seccion con el texto, y deberían tener una columna con
los códigos.

In [3]:
indicadores_estandarizados = estandarizar_df_ine(indicadores, "Indicadores demográficos")
print(indicadores_estandarizados.info())
indicadores_estandarizados.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 224973 entries, 126 to 515591
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   Provincia  224973 non-null  object
 1   CMuni      224973 non-null  object
 2   CUSEC      224973 non-null  object
 3   Indicador  224973 non-null  object
 4   Periodo    224973 non-null  int64 
 5   Total      198755 non-null  object
dtypes: int64(1), object(5)
memory usage: 12.0+ MB
None


,Provincia,CMuni,CUSEC,Indicador,Periodo,Total
179764,Leon,24165,2416501001,Porcentaje de población de 65 y más años,2016,43
403653,Valladolid,47028,4702801001,Porcentaje de población menor de 18 años,2020,"19,5"
215010,Palencia,34120,3412005015,Porcentaje de población española,2023,"96,3"
507623,Zamora,49248,4924801001,Tamaño medio del hogar,2018,"2,3"
460296,Valladolid,47217,4721701001,Porcentaje de población de 65 y más años,2023,18


## Filtrado de años
Revisando la documentación del INE, en el año 2021 se cambió radicalemente la metodología que define las secciones censales,
y en concreto en Castilla y León se aumentó el numero de censos de 2700 a unos 3500 apróximadamente. Es por ello que, si bien
se dispone de datos de años anteriores, sería complejo y peligroso fragmentar y proyectar los censos de años previos en la malla 
censal actual, por lo que se filtraran datos de años previos

In [4]:
# Reviso los indicadores disponibles
revisar_indicadores_disponibles(indicadores_estandarizados)

📅 Años disponibles:
[2023 2022 2021 2020 2019 2018 2017 2016 2015]
------------------------------------------------------------
🧩 Indicadores demográficos disponibles:
  - Edad media de la población
  - Porcentaje de población menor de 18 años
  - Porcentaje de población de 65 y más años
  - Tamaño medio del hogar
  - Porcentaje de hogares unipersonales
  - Población
  - Porcentaje de población española
------------------------------------------------------------


In [5]:
# Filtro por los años 2021 - 2023.
indicadores_recientes = indicadores_estandarizados[
    indicadores_estandarizados["Periodo"].isin([2021, 2022, 2023])
].copy()

In [6]:
# Pivoto los indicadores para tener una columna por indicador y reducir las filas de la tabla
indicadores_por_seccion = pivotar_indicadores(indicadores_recientes)
print(indicadores_por_seccion.info())
indicadores_por_seccion.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10579 entries, 0 to 10578
Data columns (total 11 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   Provincia                                 10579 non-null  object
 1   CMuni                                     10579 non-null  object
 2   CUSEC                                     10579 non-null  object
 3   Periodo                                   10579 non-null  int64 
 4   Edad_media_de_la_población                10579 non-null  object
 5   Población                                 10579 non-null  object
 6   Porcentaje_de_hogares_unipersonales       10579 non-null  object
 7   Porcentaje_de_población_de_65_y_más_años  10579 non-null  object
 8   Porcentaje_de_población_española          10579 non-null  object
 9   Porcentaje_de_población_menor_de_18_años  10579 non-null  object
 10  Tamaño_medio_del_hogar                    1057

,Provincia,CMuni,CUSEC,Periodo,Edad_media_de_la_población,Población,Porcentaje_de_hogares_unipersonales,Porcentaje_de_población_de_65_y_más_años,Porcentaje_de_población_española,Porcentaje_de_población_menor_de_18_años,Tamaño_medio_del_hogar
3370,Leon,24105,2410501001,2022,"46,3",1.892,"30,5","22,8","96,5","16,5","2,5"
4487,Palencia,34120,3412007004,2022,"41,8",1.816,"31,9","17,3","94,6","19,4","2,4"
8196,Valladolid,47068,4706801001,2022,54,36,"52,6","27,8","72,2","5,6","1,9"
4764,Palencia,34221,3422101001,2023,"50,5",183,41,"31,2",100,"11,5","2,2"
7455,Soria,42059,4205901001,2023,"61,2",21,"22,2","57,1","100,0","4,8","2,3"


In [7]:
# Voy a calcular tambien el % de extranjeros
indicadores_por_seccion["Porcentaje_de_población_española"] = (
    indicadores_por_seccion["Porcentaje_de_población_española"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)
indicadores_por_seccion["Porcentaje_de_población_extranjera"] = 100 - indicadores_por_seccion["Porcentaje_de_población_española"]

In [8]:
# Convertimos a numérico (sustituyendo comas por puntos)
cols_num = [
    "Edad_media_de_la_población",
    "Población",
    "Porcentaje_de_hogares_unipersonales",
    "Porcentaje_de_población_española",
    "Porcentaje_de_población_extranjera",
    "Tamaño_medio_del_hogar"
]

indicadores_por_seccion[cols_num] = (
    indicadores_por_seccion[cols_num]
    .replace(",", ".", regex=True)
    .apply(pd.to_numeric, errors="coerce")
)

# --- 1️⃣ Nivel MUNICIPAL ---
indicadores_por_municipio = (
    indicadores_por_seccion
    .groupby(["Provincia", "CMuni", "Periodo"], as_index=False)
    .apply(lambda g: pd.Series({
        "Edad_media_de_la_población": (g["Edad_media_de_la_población"] * g["Población"]).sum() / g["Población"].sum(),
        "Porcentaje_de_hogares_unipersonales": (g["Porcentaje_de_hogares_unipersonales"] * g["Población"]).sum() / g["Población"].sum(),
        "Porcentaje_de_población_española": (g["Porcentaje_de_población_española"] * g["Población"]).sum() / g["Población"].sum(),
        "Porcentaje_de_población_extranjera": (g["Porcentaje_de_población_extranjera"] * g["Población"]).sum() / g["Población"].sum(),
        "Tamaño_medio_del_hogar": (g["Tamaño_medio_del_hogar"] * g["Población"]).sum() / g["Población"].sum(),
    }))
    .reset_index(drop=True)
)

# --- 2️⃣ Nivel PROVINCIAL ---
indicadores_por_provincia = (
    indicadores_por_seccion
    .groupby(["Provincia", "Periodo"], as_index=False)
    .apply(lambda g: pd.Series({
        "Edad_media_de_la_población": (g["Edad_media_de_la_población"] * g["Población"]).sum() / g["Población"].sum(),
        "Porcentaje_de_hogares_unipersonales": (g["Porcentaje_de_hogares_unipersonales"] * g["Población"]).sum() / g["Población"].sum(),
        "Porcentaje_de_población_española": (g["Porcentaje_de_población_española"] * g["Población"]).sum() / g["Población"].sum(),
        "Porcentaje_de_población_extranjera": (g["Porcentaje_de_población_extranjera"] * g["Población"]).sum() / g["Población"].sum(),
        "Tamaño_medio_del_hogar": (g["Tamaño_medio_del_hogar"] * g["Población"]).sum() / g["Población"].sum(),
    }))
    .reset_index(drop=True)
)

# Elimino también las columnas de edades ya que los datos se sacarán del censo anual de población
indicadores_por_seccion = indicadores_por_seccion.drop(
    columns=["Población","Porcentaje_de_población_de_65_y_más_años", "Porcentaje_de_población_menor_de_18_años"])

C:\Users\Carlo\AppData\Local\Temp\ipykernel_26332\1032388972.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
C:\Users\Carlo\AppData\Local\Temp\ipykernel_26332\1032388972.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


# Export de los resultados

In [9]:
# Creamos la carpeta si no existe
os.makedirs(DATA_OUTPUTS_DD, exist_ok=True)

# Rutas de salida
ruta_seccion = os.path.join(DATA_OUTPUTS_DD, "indicadores_demográficos_por_seccion.csv")
ruta_municipio = os.path.join(DATA_OUTPUTS_DD, "indicadores_demográficos_por_municipio.csv")
ruta_provincia = os.path.join(DATA_OUTPUTS_DD, "indicadores_demográficos_por_provincia.csv")

# Guardar DataFrames
indicadores_por_seccion.to_csv(
    ruta_seccion,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
indicadores_por_municipio.to_csv(
    ruta_municipio,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
indicadores_por_provincia.to_csv(
    ruta_provincia,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
print(f"✅ Archivos guardados correctamente en: {DATA_OUTPUTS_DD}")

✅ Archivos guardados correctamente en: D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\DD_Dim_demografica
